# Min Cluster Size Parameter Sweep

This notebook tests how `min_cluster_size` (5 to 50, step 5) affects:
- Number of topics found
- Number of outlier documents (topic = -1)
- Percentage of documents classified as outliers

All other HDBSCAN and UMAP parameters are kept constant.

## Setup

In [1]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from nltk.corpus import stopwords
import nltk
from pathlib import Path
import pickle
import plotly.graph_objects as go
from plotly.subplots import make_subplots

C:\Users\sile9\anaconda3\envs\bertopic_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configure Paths

In [2]:
project_root = Path.cwd().parent if "Scripts" in Path.cwd().parts else Path.cwd()
data_dir = project_root / 'Data'
output_dir = project_root / 'Outputs'
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Project root: {project_root}")

Project root: c:\Users\sile9\Documents\projects\parldebates_analysis


## Load Documents & Embeddings

In [3]:
df = pd.read_csv(data_dir / '05_transcripts_climate_for_topic_modeling.csv')
text_column = 'transcript_text'
documents = df[text_column].tolist()
print(f"Loaded {len(documents)} documents")

# Load or compute embeddings
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb_path = data_dir / '06_embeddings.pickle'

if emb_path.exists():
    print("Loading existing embeddings from disk...")
    with emb_path.open('rb') as handle:
        embeddings = pickle.load(handle)
else:
    print("Computing embeddings...")
    embeddings = embedding_model.encode(documents, show_progress_bar=True)
    emb_path.parent.mkdir(parents=True, exist_ok=True)
    with emb_path.open('wb') as handle:
        pickle.dump(embeddings, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Embeddings shape: {embeddings.shape}")

Loaded 4987 documents


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4597.78it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading existing embeddings from disk...
Embeddings shape: (4987, 768)


## Prepare Shared Components

UMAP, vectorizer, and representation models are shared across all runs — only HDBSCAN's `min_cluster_size` varies.

In [4]:
# Stopwords
nltk.download('stopwords', quiet=True)
german_stopwords = set(stopwords.words('german'))
french_stopwords = set(stopwords.words('french'))
swiss_german_stopwords = {'dass'}
all_stopwords = german_stopwords.union(french_stopwords).union(swiss_german_stopwords)

# Vectorizer
vectorizer_model = CountVectorizer(
    min_df=2,
    max_df=0.85,
    ngram_range=(1, 2),
    stop_words=list(all_stopwords)
)

# Representation models (without OpenAI for speed)
keybert = KeyBERTInspired()
mmr = MaximalMarginalRelevance(diversity=0.3)
representation_model = {
    "KeyBERT": keybert,
    "MMR": mmr,
}

print("Shared components ready.")

Shared components ready.


## Run Parameter Sweep

Loop over `min_cluster_size` from 5 to 50 in steps of 5.

In [5]:
min_cluster_sizes = list(range(5, 51, 5))
results = []

n_docs = len(documents)

for mcs in min_cluster_sizes:
    print(f"\n{'='*50}")
    print(f"Testing min_cluster_size = {mcs}")
    print(f"{'='*50}")
    
    # Fresh UMAP and HDBSCAN for each run
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )
    
    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=mcs,
        cluster_selection_epsilon=0.0,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )
    
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
        nr_topics="auto",
        top_n_words=10,
        verbose=False
    )
    
    topics, probs = topic_model.fit_transform(documents, embeddings)
    
    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    n_outliers = sum(1 for t in topics if t == -1)
    pct_outliers = round(n_outliers / n_docs * 100, 1)
    
    results.append({
        'min_cluster_size': mcs,
        'n_topics': n_topics,
        'n_outliers': n_outliers,
        'pct_outliers': pct_outliers
    })
    
    print(f"  Topics: {n_topics} | Outliers: {n_outliers} ({pct_outliers}%)")

print("\nSweep complete!")


Testing min_cluster_size = 5
  Topics: 41 | Outliers: 1937 (38.8%)

Testing min_cluster_size = 10
  Topics: 33 | Outliers: 2017 (40.4%)

Testing min_cluster_size = 15
  Topics: 24 | Outliers: 2136 (42.8%)

Testing min_cluster_size = 20
  Topics: 15 | Outliers: 1583 (31.7%)

Testing min_cluster_size = 25
  Topics: 17 | Outliers: 1359 (27.3%)

Testing min_cluster_size = 30
  Topics: 16 | Outliers: 1490 (29.9%)

Testing min_cluster_size = 35
  Topics: 13 | Outliers: 1574 (31.6%)

Testing min_cluster_size = 40
  Topics: 7 | Outliers: 903 (18.1%)

Testing min_cluster_size = 45
  Topics: 7 | Outliers: 821 (16.5%)

Testing min_cluster_size = 50
  Topics: 7 | Outliers: 860 (17.2%)

Sweep complete!


## Results Table

In [6]:
results_df = pd.DataFrame(results)
results_df.columns = ['min_cluster_size', 'Number of Topics', 'Number of Outliers', 'Outliers (%)']
print(results_df.to_string(index=False))

# Save to CSV
results_df.to_csv(data_dir / '06b_min_cluster_size_sweep.csv', index=False)
print(f"\nResults saved to {data_dir / '06b_min_cluster_size_sweep.csv'}")

 min_cluster_size  Number of Topics  Number of Outliers  Outliers (%)
                5                41                1937          38.8
               10                33                2017          40.4
               15                24                2136          42.8
               20                15                1583          31.7
               25                17                1359          27.3
               30                16                1490          29.9
               35                13                1574          31.6
               40                 7                 903          18.1
               45                 7                 821          16.5
               50                 7                 860          17.2

Results saved to c:\Users\sile9\Documents\projects\parldebates_analysis\Data\06b_min_cluster_size_sweep.csv
